In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
from torch.autograd import Variable
from torch.optim import Adam
import torch.nn.functional as F
from torch.nn.functional import relu,tanh
from torchvision.datasets.mnist import MNIST
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd
import random
import csv
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
data=np.genfromtxt('/export/scratch2/sumanta/FiRE/data/preprocessedData_jurkat_two_species_1580.txt')
#label=np.genfromtxt('cell_type_no.csv',delimiter=',').astype('int64')
labels = np.genfromtxt('/export/scratch2/sumanta/FiRE/data/labels_jurkat_two_species_1580.txt', dtype=np.int) 
labels=labels-1
top_k=data.shape[1]
#label=np.genfromtxt('cell_type_no_onlyrna.csv',delimiter=',').astype('int64')

In [ ]:
data=pd.DataFrame(data)

In [ ]:
data.shape

In [ ]:
genes = np.arange(1, data.shape[1]+1)

In [ ]:
len(np.where(labels==1)[0])

In [ ]:
class ConvCaps2D(nn.Module):
    def __init__(self):
        super(ConvCaps2D, self).__init__()
        # The paper suggests having 32 8D capsules
        self.capsules = nn.ModuleList([nn.Conv2d(in_channels = 1, out_channels = 8, kernel_size=(1,5), stride=2)
                                       for _ in range(32)])
        
    def squash(self, tensor, dim=-1):
        norm = (tensor**2).sum(dim=dim, keepdim = True) # norm.size() is (None, 1152, 1)
        scale = norm / (1 + norm) # scale.size()  is (None, 1152, 1)  
        return scale*tensor / torch.sqrt(norm)
        
    def forward(self, x):
        outputs = [capsule(x).view(x.size(0), 8, -1) for capsule in self.capsules] # 32 list of (None, 1, 8, 36)
        outputs = torch.cat(outputs, dim = 2).permute(0, 2, 1)  # outputs.size() is (None, 1152, 8)
        return self.squash(outputs)

In [ ]:
class Caps1D(nn.Module):
    def __init__(self):
        super(Caps1D, self).__init__()
        self.num_caps = 2
        self.num_iterations = 3
        self.W = nn.Parameter(torch.randn(2, 2336, 8, 16))
        
    def softmax(self, x, dim = 1):
        transposed_input = x.transpose(dim, len(x.size()) - 1)
        softmaxed_output = F.softmax(transposed_input.contiguous().view(-1, transposed_input.size(-1)))
        return softmaxed_output.view(*transposed_input.size()).transpose(dim, len(x.size()) - 1)

    def squash(self, tensor, dim=-1):
        norm = (tensor**2).sum(dim=dim, keepdim = True) # norm.size() is (None, 1152, 1)
        scale = norm / (1 + norm)        
        return scale*tensor / torch.sqrt(norm)
   
    # Routing algorithm
    def forward(self, u):
        # u.size() is (None, 1152, 8)
        '''
        From documentation
        For example, if tensor1 is a j x 1 x n x m Tensor and tensor2 is a k x m x p Tensor, 
        out will be an j x k x n x p Tensor.
        
        We need j = None, 1, n = 1152, k = 10, m = 8, p = 16
        '''
        
        u_ji = torch.matmul(u[:, None, :, None, :], self.W) # u_ji.size() is (None, 10, 1152, 1, 16)
        
        b = Variable(torch.zeros(u_ji.size())) # b.size() is (None, 10, 1152, 1, 16)
        
        for i in range(self.num_iterations):
            c = self.softmax(b, dim=2)
            v = self.squash((c * u_ji).sum(dim=2, keepdim=True)) # v.size() is (None, 10, 1, 1, 16)

            if i != self.num_iterations - 1:
                delta_b = (u_ji * v).sum(dim=-1, keepdim=True)
                b = b + delta_b
        
        # Now we simply compute the length of the vectors and take the softmax to get probability.
        v = v.squeeze()
     #   print(v.shape)
        y=v.data.cpu().numpy()
        hook1=y[:,0:2,0:16]
    #    print(y.shape)
        y = np.reshape(y,(len(y)*2,16))
   #     print(y.shape)
        with open('test_16.csv', 'w') as outfile:
    #        for slice_2d in x:
           writer=csv.writer(outfile, delimiter='\t')
           writer.writerows(y)
        classes = (v ** 2).sum(dim=-1) ** 0.5
       # print(classes.shape)
        classes = F.softmax(classes) # This is not done in the paper, but I've done this to use CrossEntropyLoss.
      #  print(classes.shape)
        hook=c.data.cpu().numpy()
      #  print(hook.shape)
        #from numpy import savetxt
        #savetxt('test.csv', hook[0:4][0:13,0:928,-1,-1], delimiter=',')
        #with open('test.csv', 'w', newline='') as csvfile:
        #    writer=csv.writer(csvfile, delimiter='\t')
        #    writer.writerows(hook[0:4][0:13,0:928,-1,-1])
        x=hook[:,0:2,0:2336,-1,-1]
       # print(x.shape)
        # x=x.flatten()
        x = np.reshape(x,(len(x)*2,2336))
        print(x.shape)
        with open('test.csv', 'w') as outfile:
    #        for slice_2d in x:
           writer=csv.writer(outfile, delimiter='\t')
           writer.writerows(x)
               #np.savetxt(outfile, slice_2d)
        return classes
net = Caps1D()

In [ ]:
class CapsNet(nn.Module):
    def __init__(self):
        super(CapsNet, self).__init__()
        
        #self.conv1 = nn.Conv2d(in_channels = 1, out_channels = 256, kernel_size = (1,4), stride = 1)
        self.fc1 = nn.Linear(top_k,150)
        self.dropout1 = nn.Dropout(p=0.5)
        self.primaryCaps = ConvCaps2D()
        self.digitCaps = Caps1D()
        
        
    def forward(self, x):
        x = relu(self.dropout1(self.fc1(x)))#F.relu(self.conv1(x))
        x = self.primaryCaps(x)
        x = self.digitCaps(x)
        
        return x

net = CapsNet()

In [ ]:
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters())

In [ ]:
def evaluate(model, X, Y, batch_size = 50):
    results = []
    predicted = []
    for i in range(len(X)//batch_size):
        s = i*batch_size
        e = i*batch_size+batch_size
        
        inputs = Variable(torch.from_numpy(X[s:e]))
        pred = net(inputs)
        
        predicted += list(np.argmax(pred.data.cpu().numpy(), axis = 1))

    Y=Y[0:len(predicted)]
  #  acc = sum(Y == predicted)*1.0/(len(Y)) 
    f1=f1_score(Y, predicted)
    return f1

In [ ]:
###train test split and validtion with f1-score
batch_size=128
trn_acc = []
tst_acc = []
trn_loss =[]
tst_loss=[]
f1_trn=[]
f1_tst=[]
#net=CapsNet()
#data1=torch.from_numpy(data)
#fc1 = nn.Linear(top_k,350)
#dropout1 = nn.Dropout(p=0.5)

#data1=F.relu(dropout1(fc1(data1.float())))
#data2=data1.cpu().detach().numpy()
#data2=pd.DataFrame(data)
data=pd.DataFrame(data)
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.1)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2)

d=X_train.values.reshape(len(X_train),1, 1000, order='F')
d1_train=np.expand_dims(d.astype('float32'), 1)
        #l1_train=y_train
indices = np.random.permutation(len(d1_train))
d1_train=d1_train[indices]
l1_train=y_train[indices]

d1=X_val.values.reshape(len(X_val),1, 1000, order='F')
d1_test=np.expand_dims(d1.astype('float32'), 1)
      # l1_test=y_test
indices = np.random.permutation(len(d1_test))
d1_test=d1_test[indices]
l1_test=y_val[indices]
#liveloss = PlotLosses()


#logs = {}
loss_trn=[]
loss_test=[]
    #net1=net

for epoch in range(120):  # 500 epochs
    for phase in ['train', 'validation']:
        if phase == 'train':
            running_loss=0 
            for i in range(len(d1_train)//batch_size-1):    ##iteration
                print(i,)
                s = i*batch_size
                e = i*batch_size+batch_size

                inputs = torch.from_numpy(d1_train[s:e])
                labels = torch.LongTensor(np.array(l1_train[s:e]))

                    # wrap them in Variable
                inputs, labels = Variable(inputs), Variable(labels)

                    # zero the parameter gradients
                optimizer.zero_grad()

                    # forward + backward + optimize
                outputs = net(inputs)

                loss = criterion(outputs, labels)
                loss.backward()

                optimizer.step()
                running_loss += loss.data.item()
        #    print("Epoch, Loss - {}, {}".format(i, running_loss))
                del inputs, labels
                #print('\n')
               # trn_loss.append(running_loss)
        else: 
            #r=random.sample(range(1, len(d1_train)), 1000)
            #trn_acc.append(evaluate(net, d1_train[r], l1_train[r], batch_size = 128)) 
            #tst_acc.append(evaluate(net, d1_test, l1_test, batch_size=128)) 
            f1_trn.append(evaluate(net, d1_train, l1_train, batch_size = 128))
            f1_tst.append(evaluate(net, d1_test, l1_test, batch_size=128))
            
            #out_train=net(torch.from_numpy(d1_train[r]))
            #out_test=net(torch.from_numpy(d1_test))
           # loss_trn = criterion(out_train, torch.LongTensor(np.array(l1_train[r])))
           # loss_test= criterion(out_test, torch.LongTensor(np.array(l1_test)))
            #trn_loss.append(loss_trn.data.item())
            #tst_loss.append(loss_test.data.item())
            print("f1_score train",f1_trn)
            print("f1_score_test",f1_tst)
            #print("train_loss",trn_loss)
            #print("test_acc",tst_loss)
            #logs['log_loss_trn'] = loss_trn.append(loss_trn)
        #logs['log_loss_tst'] = loss_test.append(loss_test)
        #logs['tr_accuracy'] = trn_acc[-1]
        #logs['tst_accuracy'] = tst_acc[-1]

    #liveloss.update(logs)
    #liveloss.draw()
    #print("Epoch, Loss - {}, {}".format(epoch, running_loss))
    #print("Train - ", trn_acc[-1])
    #print("Test - ", tst_acc[-1])

In [ ]:
plt.plot(range(0, 120), f1_trn)
plt.title('f1 score training set')
plt.show()

#plt.plot(range(0, epoch), trn_loss)
#plt.title('Training loss')
#plt.show()

plt.plot(range(0, 120), f1_tst)
plt.title('f1 score test set')
plt.show()

In [ ]:
torch.save(net,'capsnetmodel_jurkat.pt')

In [ ]:
net = torch.load('capsnetmodel_jurkat.pt')
net.eval()

In [ ]:
import dill

model_copy=dill.dumps(net)
torch.save(model_copy,'model_caps_jurkat.pt')

#model1 = torch.load('model_ignite_original.pt')
#model=dill.loads(model1)

In [ ]:
import dill
model1 = torch.load('model_caps_jurkat.pt')
model=dill.loads(model1)
net=model

In [ ]:
data=pd.DataFrame(data)
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.1)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2)

d=X_train.values.reshape(len(X_train),1, 1000, order='F')
d1_train=np.expand_dims(d.astype('float32'), 1)
        #l1_train=y_train
indices = np.random.permutation(len(d1_train))
d1_train=d1_train[indices]
l1_train=y_train[indices]

d1=X_val.values.reshape(len(X_val),1, 1000, order='F')
d1_test=np.expand_dims(d1.astype('float32'), 1)
      # l1_test=y_test
indices = np.random.permutation(len(d1_test))
d1_test=d1_test[indices]
l1_test=y_val[indices]
evaluate(net, d1_train, l1_train, batch_size=d1_train.shape[0])
np.savetxt('org_label.txt',l1_train,delimiter=',')

In [ ]:
evaluate(net,d1_test , l1_test, batch_size=d1_test.shape[0])
#np.savetxt('org_label.txt',l1_test,delimiter=',')

In [ ]:
test_cell= pd.read_csv("test.csv", sep='\t',header=None)
np.savetxt('test_cell_coupling.csv',test_cell,delimiter=',')


In [ ]:
np.savetxt('d1_test.txt',d1_test[0:d1_test.shape[0],0,0,0:1000],delimiter=',')

In [ ]:
np.savetxt('org_label.txt',l1_test,delimiter=',')

In [ ]:
#save the model 

torch.save(net.state_dict(),'capsnet_jurkat.pt')

In [ ]:
##for loading 
checkpt=torch.load('capsnet_jurkat.pt')
net.load_state_dict(checkpt)

In [ ]:
data=np.genfromtxt('/export/scratch2/sumanta/FiRE/data/preprocessedData_jurkat_two_species_1580.txt')
data=pd.DataFrame(data)
colnames=data.columns.values

In [ ]:
n=np.where(labels==1)

In [ ]:
data=np.genfromtxt('/export/scratch2/sumanta/FiRE/data/preprocessedData_jurkat_two_species_1580.txt')
data=pd.DataFrame(data)
colnames=data.columns.values
p=3
data.loc[:, np.setdiff1d(colnames,colnames[0:p])]=0

In [ ]:
data=np.asarray(data)

In [ ]:
n=np.where(labels==0)
d=data[n[0],:]
d=d.reshape(len(d),1, 1000)
d=np.expand_dims(d.astype('float32'), 1)
net(Variable(torch.from_numpy(d)))
test_cell= pd.read_csv("test.csv", sep='\t',header=None)
test_cell=test_cell.to_numpy()
#test_cell=log(test_cell/(1-test_cell))
test_cell_reformated=np.zeros(shape=(test_cell.shape[0],32))

In [ ]:
for i in range(np.shape(test_cell)[0]):
            k=0
            for j in range(32):
                test_cell_reformated[i,j]= np.amax(test_cell[i,k:(j+1)*73])
                k=(j+1)*73+1

In [ ]:
r=range(0,test_cell_reformated.shape[0],2)

In [ ]:
n[0].shape[0]

In [ ]:
avg_s_l1=np.zeros(shape=(2,32))
s=test_cell_reformated[0,:]
s1=test_cell_reformated[1,:]
for i in range(0,test_cell_reformated.shape[0],2):
    s=np.add(s,test_cell_reformated[i,:])
    s1=np.add(s1,test_cell_reformated[(i+1),:])
    avg_s_l1=np.vstack((s/n[0].shape[0],s1/n[0].shape[0]))

In [ ]:
coup_gene=np.zeros(shape=(1,32))
coup_gene=np.vstack((coup_gene,avg_s_l1))

In [ ]:
coup_gene=np.vstack((coup_gene,avg_s_l1))

In [ ]:
coup_gene_final=np.zeros(shape=(1,32))
coup_gene_final=coup_gene[0:2]+coup_gene[2:4]

In [ ]:
r=range(100,200,1)

In [ ]:
r[99]

In [ ]:
###get the coupling coefficient for the primary capsule i
coup_gene_all=np.zeros(shape=(1,32))
for p in range(500,1000,1):
    print("p is ", p)
    data=np.genfromtxt('/export/scratch2/sumanta/FiRE/data/preprocessedData_jurkat_two_species_1580.txt')
    data=pd.DataFrame(data)
    colnames=data.columns.values
    data.loc[:, np.setdiff1d(colnames,colnames[0:p])]=0
    coup_gene=np.zeros(shape=(1,32))
    coup_gene_final=np.zeros(shape=(1,32))
    for l in range(2):
        n=np.where(labels==l)
        data=np.asarray(data)
        d=data[n[0],:]
        d=d.reshape(len(d),1, 1000)
        d=np.expand_dims(d.astype('float32'), 1)
        net(Variable(torch.from_numpy(d)))
        test_cell= pd.read_csv("test.csv", sep='\t',header=None)
        test_cell=test_cell.to_numpy()
            #test_cell=log(test_cell/(1-test_cell))
        test_cell_reformated=np.zeros(shape=(test_cell.shape[0],32))
        for i in range(np.shape(test_cell)[0]):
            k=0
            for j in range(32):
                test_cell_reformated[i,j]= np.amax(test_cell[i,k:(j+1)*73])
                k=(j+1)*73+1
        avg_s_l1=np.zeros(shape=(2,32))
        s=test_cell_reformated[0,:]
        s1=test_cell_reformated[1,:]
        for i in range(0,test_cell_reformated.shape[0],2):
            s=np.add(s,test_cell_reformated[i,:])
            s1=np.add(s1,test_cell_reformated[(i+1),:])
        avg_s_l1=np.vstack((s/n[0].shape[0],s1/n[0].shape[0]))
        coup_gene=np.vstack((coup_gene,avg_s_l1))
    coup_gene=coup_gene[1:5,:]
    #coup_gene_all.shape
    #np.savetxt('coup_gene_all.csv',coup_gene_all,delimiter=',')
    coup_gene_final=coup_gene[0:2]+coup_gene[2:4]
    coup_gene_all=np.vstack((coup_gene_all,coup_gene_final))
    
np.savetxt('coup_gene_all_jrcat.csv',coup_gene_all,delimiter=',')

In [ ]:
coup_gene_all.shape

In [ ]:
coup_gene_final=coup_gene[0:2]+coup_gene[2:4]
    coup_gene_all=np.vstack((coup_gene_all,coup_gene_final)

In [ ]:
### testing by varying the jurcat cell proportion in training data, test data is same

batch_size=128
trn_acc = []
tst_acc = []
trn_loss =[]
tst_loss=[]
f1_trn=[]
f1_tst=[]
#net=CapsNet()
#data1=torch.from_numpy(data)
#fc1 = nn.Linear(top_k,350)
#dropout1 = nn.Dropout(p=0.5)

#data1=F.relu(dropout1(fc1(data1.float())))
#data2=data1.cpu().detach().numpy()
#data2=pd.DataFrame(data)
#data=pd.DataFrame(data)
skf = StratifiedShuffleSplit(n_splits=2, test_size=0.2, train_size=0.8,random_state=0)
#X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.1,stratify=1)

def weights_init(m):
    if isinstance(m, nn.Conv2d):
        torch.nn.init.xavier_uniform(m.weight)
        m.bias.data.fill_(0.01)
    #net1=net
for train, test in skf.split(data, labels):
    
   # X_train=data[train,:]
   # y_train=labels[train]
    f1_tst=[]
    for p in range(1):#range(100,0,-10):
        p=40
        X_train=data[train,:]
        y_train=labels[train]
        s=round(p*len(np.where(y_train==1)[0])/100)
        l=np.where(labels[train]==1)[0]
        X_train=X_train[np.setdiff1d(range(len(y_train)),l[0:s]),:]
        y_train=y_train[np.setdiff1d(range(len(y_train)),l[0:s])]
        X_train=pd.DataFrame(X_train)
        d=X_train.values.reshape(len(X_train),1, 1000, order='F')
        d1_train=np.expand_dims(d.astype('float32'), 1)
        indices = np.random.permutation(len(d1_train))
        d1_train=d1_train[indices]
        l1_train=y_train[indices]

        X_test=data[test,:]
        y_test=labels[test]
        X_test=pd.DataFrame(X_test)
        d1=X_test.values.reshape(len(X_test),1, 1000, order='F')
        d1_test=np.expand_dims(d1.astype('float32'), 1)
          # l1_test=y_test
        indices = np.random.permutation(len(d1_test))
        d1_test=d1_test[indices]
        l1_test=y_test[indices]
        f1_trn=[]
        for epoch in range(100):  # 500 epochs
            for phase in ['train', 'validation']:
                if phase == 'train':
                    running_loss=0 
                    for i in range(len(d1_train)//batch_size-1):    ##iteration
                        print(i,)
                        s = i*batch_size
                        e = i*batch_size+batch_size

                        inputs = torch.from_numpy(d1_train[s:e])
                        label = torch.LongTensor(np.array(l1_train[s:e]))
                        inputs, label = Variable(inputs), Variable(label)
                        optimizer.zero_grad()
                        outputs = net(inputs)
                        loss = criterion(outputs, label)
                        loss.backward()
                        optimizer.step()
                        running_loss += loss.data.item()
                        del inputs, label
                else: 
                    r=random.sample(range(1, len(d1_train)), 500)
                    f1_trn.append(evaluate(net, d1_train[r], l1_train[r], batch_size = 128))
                    print("f1_score train",f1_trn)
                    
        f1_tst.append(evaluate(net, d1_test, l1_test, batch_size=128))
        print("f1_score test",f1_tst)
        net.apply(weights_init)    
            #liveloss.update(logs)
            #liveloss.draw()
            #print("Epoch, Loss - {}, {}".format(epoch, running_loss))
            #print("Train - ", trn_acc[-1])
            #print("Test - ", tst_acc[-1])

In [ ]:
###get the data with 20%, 40%...100% inclusion of jurkat cell samples
skf = StratifiedShuffleSplit(n_splits=2, test_size=0.2, train_size=0.8,random_state=0)
for train, test in skf.split(data, labels):
    for p in range(120,0,-20):
            #p=40
        X_train=data[train,:]
        y_train=labels[train]
        s=round(p*len(np.where(y_train==1)[0])/100)
        l=np.where(labels[train]==1)[0]
        X_train=X_train[np.setdiff1d(range(len(y_train)),l[0:s]),:]
        y_train=y_train[np.setdiff1d(range(len(y_train)),l[0:s])]
        p1=100-p
        np.savetxt("trainset."+str(p1)+".txt", X_train)
        np.savetxt("trainlabel."+str(p1)+".txt", y_train)

np.savetxt("test_set.txt", data[test,:])
np.savetxt("test_set_label.txt", labels[test])


In [ ]:
a=[ 0.5, 0.5, 0.6, 0.9090909090909091, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

In [ ]:
np.mean(a)

In [ ]:
### testing by train test data split with stratefied sampling

batch_size=128
trn_acc = []
tst_acc = []
trn_loss =[]
tst_loss=[]
f1_trn=[]
f1_tst=[]
#net=CapsNet()
#data1=torch.from_numpy(data)
#fc1 = nn.Linear(top_k,350)
#dropout1 = nn.Dropout(p=0.5)

#data1=F.relu(dropout1(fc1(data1.float())))
#data2=data1.cpu().detach().numpy()
#data2=pd.DataFrame(data)
#data=pd.DataFrame(data)
skf = StratifiedShuffleSplit(n_splits=2, test_size=0.2, train_size=0.8,random_state=0)
#X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.1,stratify=1)


    #net1=net
for train, test in skf.split(data, labels):
    
    X_train=data[train,:]
    y_train=labels[train]
    X_train=pd.DataFrame(X_train)
    d=X_train.values.reshape(len(X_train),1, 1000, order='F')
    d1_train=np.expand_dims(d.astype('float32'), 1)
    indices = np.random.permutation(len(d1_train))
    d1_train=d1_train[indices]
    l1_train=y_train[indices]
    
    X_test=data[test,:]
    y_test=labels[test]
    X_test=pd.DataFrame(X_test)
    d1=X_test.values.reshape(len(X_test),1, 1000, order='F')
    d1_test=np.expand_dims(d1.astype('float32'), 1)
      # l1_test=y_test
    indices = np.random.permutation(len(d1_test))
    d1_test=d1_test[indices]
    l1_test=y_test[indices]
    f1_trn=[]
    f1_tst=[]
    for epoch in range(100):  # 500 epochs
        for phase in ['train', 'validation']:
            if phase == 'train':
                running_loss=0 
                for i in range(len(d1_train)//batch_size-1):    ##iteration
                    print(i,)
                    s = i*batch_size
                    e = i*batch_size+batch_size

                    inputs = torch.from_numpy(d1_train[s:e])
                    label = torch.LongTensor(np.array(l1_train[s:e]))

                        # wrap them in Variable
                    inputs, label = Variable(inputs), Variable(label)

                        # zero the parameter gradients
                    optimizer.zero_grad()

                        # forward + backward + optimize
                    outputs = net(inputs)

                    loss = criterion(outputs, label)
                    loss.backward()

                    optimizer.step()
                    running_loss += loss.data.item()
            #    print("Epoch, Loss - {}, {}".format(i, running_loss))
                    del inputs, label
                    #print('\n')
                   # trn_loss.append(running_loss)
            else: 
                r=random.sample(range(1, len(d1_train)), 500)
                #trn_acc.append(evaluate(net, d1_train[r], l1_train[r], batch_size = 128)) 
                #tst_acc.append(evaluate(net, d1_test, l1_test, batch_size=128)) 
                f1_trn.append(evaluate(net, d1_train[r], l1_train[r], batch_size = 128))
                f1_tst.append(evaluate(net, d1_test, l1_test, batch_size=128))

                #out_train=net(torch.from_numpy(d1_train[r]))
                #out_test=net(torch.from_numpy(d1_test))
               # loss_trn = criterion(out_train, torch.LongTensor(np.array(l1_train[r])))
               # loss_test= criterion(out_test, torch.LongTensor(np.array(l1_test)))
                #trn_loss.append(loss_trn.data.item())
                #tst_loss.append(loss_test.data.item())
                print("f1_score train",f1_trn)
                print("f1_score_test",f1_tst)
                #print("train_loss",trn_loss)
                #print("test_acc",tst_loss)
                #logs['log_loss_trn'] = loss_trn.append(loss_trn)
            #logs['log_loss_tst'] = loss_test.append(loss_test)
            #logs['tr_accuracy'] = trn_acc[-1]
            #logs['tst_accuracy'] = tst_acc[-1]

        #liveloss.update(logs)
        #liveloss.draw()
        #print("Epoch, Loss - {}, {}".format(epoch, running_loss))
        #print("Train - ", trn_acc[-1])
        #print("Test - ", tst_acc[-1])

In [ ]:
from numpy import savetxt
savetxt('pred_label.csv', pred_label, delimiter=',')
savetxt('org_label.csv', l1_test, delimiter=',')


In [ ]:
plt.plot(range(0, 100), f1_trn)
plt.title('f1 score training set')
plt.show()

#plt.plot(range(0, epoch), trn_loss)
#plt.title('Training loss')
#plt.show()

plt.plot(range(0, 100), f1_tst)
plt.title('f1 score test set')
plt.show()

In [ ]:
#####for 10 fold cv with stratified sampling.  


def weights_init(m):
    if isinstance(m, nn.Linear):
        torch.nn.init.xavier_uniform(m.weight)
        m.bias.data.fill_(0.01)


data=pd.DataFrame(data)
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.1)
#main_d=X_train.values.reshape(len(X_train),1, 64, order='F')
#main_d_train=np.expand_dims(main_d.astype('float32'), 1)

d1=X_test.values.reshape(len(X_test),1, 1000, order='F')
d1_test=np.expand_dims(d1.astype('float32'), 1)
l1_test=y_test
f1_trn_per=[]
f1_vld_per=[]
f1_test=[]
for per in np.linspace(1,1,1):
    print('percentage of train:', per*100)
    indices = np.random.permutation(len(X_train))
    
    ## take 5 %  of training samples
    patch=round(len(indices)*per.astype('float32'))
    batch_dtrain=np.array(X_train)[indices[0:patch]]
    batch_ytrain=np.array(y_train)[indices[0:patch]]
    #X_train, X_test, y_train, y_test = train_test_split(data, label, test_size=0.1)
    cv = StratifiedKFold(n_splits=10, random_state=42, shuffle=False)

    batch_size=50
    f1_trn_cv=[]
    f1_vld_cv=[]
    f1_vld_perfold=[]
    f1_test=[]
    k=1
    for train_index, test_index in cv.split(batch_dtrain,batch_ytrain):
        #new_net=net
        print("fold",k)
        X_train1, X_val1, y_train1, y_val1 = batch_dtrain[train_index], batch_dtrain[test_index], batch_ytrain[train_index], batch_ytrain[test_index]
        #print(X_train[test_index].shape)
        d=X_train1.reshape(len(X_train1),1, 1000, order='F')
        d1_train=np.expand_dims(d.astype('float32'), 1)
            #l1_train=y_train
        indices = np.random.permutation(len(d1_train))
        d1_train=d1_train[indices]
        l1_train=y_train1[indices]

        d1=X_val1.reshape(len(X_val1),1, 1000, order='F')
        d1_vald=np.expand_dims(d1.astype('float32'), 1)
          # l1_test=y_test
        indices = np.random.permutation(len(d1_vald))
        d1_vald=d1_vald[indices]
        l1_vald=y_val1[indices]
       #liveloss = PlotLosses()


    #    logs = {}
        f1_trn=[]
        f1_vld=[]
         #net1=net
        #optimizer = optim.Adam(new_net.parameters())  

        for epoch in range(20):  # 500 epochs
            #net1=net
            print("epoch",epoch)
            for phase in ['train', 'validation']:
                if phase == 'train':
                    running_loss=0 
                    for i in range(len(d1_train)//batch_size-1):    ##iteration
                      #  print(i,)
                        s = i*batch_size
                        e = i*batch_size+batch_size

                        inputs = torch.from_numpy(d1_train[s:e])
                        labels = torch.LongTensor(np.array(l1_train[s:e]))

                            # wrap them in Variable
                        inputs, labels = Variable(inputs), Variable(labels)

                            # zero the parameter gradients
                        optimizer.zero_grad()

                            # forward + backward + optimize
                        outputs = net(inputs)

                        loss = criterion(outputs, labels)
                        loss.backward()

                        optimizer.step()
                        running_loss += loss.data.item()
                #    print("Epoch, Loss - {}, {}".format(i, running_loss))
                        del inputs, labels
                        #print('\n')
                       # trn_loss.append(running_loss)
                else: 
                    #r=random.sample(range(1, len(d1_train)), 100)
                  #  print("len(d1_train)",len(d1_train))
                   # print("len(l1_train)",len(l1_train))
                   # trn_acc.append(evaluate(net, d1_train, l1_train, batch_size = 50)) 
                   # tst_acc.append(evaluate(net, d1_vald, l1_vald, batch_size=50)) 
                    f1_trn.append(evaluate(net, d1_train, l1_train, batch_size = 128))
                    f1_vld.append(evaluate(net, d1_vald, l1_vald, batch_size=128))
                   # out_train=net(torch.from_numpy(d1_train))
                    #out_vald=net(torch.from_numpy(d1_vald))
                    #loss_trn = criterion(out_train, torch.LongTensor(np.array(l1_train)))
                    #loss_vald= criterion(out_vald, torch.LongTensor(np.array(l1_vald)))
                    #trn_loss.append(loss_trn.data.item())
                    #tst_loss.append(loss_vald.data.item())
                    print("f1_train",f1_trn)
                    print("f1_vald",f1_vld)
                   # print("train_loss",trn_loss)
                   # print("validation_loss",tst_loss)
                   # logs['log_loss_trn'] = loss_trn.append(loss_trn)
                   # logs['log_loss_tst'] = loss_test.append(loss_test)
                  #  logs['tr_accuracy'] = trn_acc[-1]
                  #  logs['tst_accuracy'] = tst_acc[-1]
                    #del net1
        print("kkkk")
        #del new_net   
       # trn_acc_cv.append(np.mean(trn_acc[0:20]))
       # tst_acc_cv.append(np.mean(tst_acc[0:20]))
       # trn_loss_cv.append(np.mean(trn_loss[0:20]))
       # tst_loss_cv.append(np.mean(tst_loss[0:20]))
        f1_vld_perfold.append(f1_vld)
        f1_trn_cv.append(np.mean(f1_trn[9:19]))
        f1_vld_cv.append(np.mean(f1_vld[9:19]))
        f1_test.append(evaluate(net, d1_test, l1_test, batch_size = 128))
        print("f1_train cv",f1_trn_cv)
        print("f1_vald cv",f1_vld_cv)
        print("f1_test",f1_test)
        del f1_trn, f1_vld
        k=k+1
        net.apply(weights_init)

    #f1_trn_per.append(f1_trn_cv)
    #f1_tst_per.append(f1_tst_cv)
    #tst_acc_per.append(test_acc)
    #print("f1 train for per:", per*100, f1_trn_per)
    #print("f1 tst for per:", per*100, f1_tst_per)
    #print("test acc_for per:", per*100, test_acc)


In [ ]:
from numpy import savetxt
savetxt('caps_f1score_trn_perfold.csv', f1_trn_cv, delimiter=',')
savetxt('caps_f1score_vald_perfold.csv', f1_vld_cv, delimiter=',')
savetxt('caps_f1score_vald_allfold.csv', f1_vld_perfold, delimiter=',')
savetxt('caps_f1score_test_perfold.csv', f1_test, delimiter=',')
#savetxt('cnn_rnameth_valdacc_per.csv',vald_acc_per, delimiter=',')

In [ ]:
###validation and test f1 score for different percentage of samples 

#def weights_init(m):
#    if isinstance(m, nn.Linear):
#        torch.nn.init.xavier_uniform(m.weight)
#        m.bias.data.fill_(0.01)
#    elif isinstance(m, nn.BatchNorm2d):
#        m.weight.data.normal_(1.0, 0.02)
#        m.bias.data.fill_(0)   


def init_weights(m):
    if type(m) in [nn.Conv2d]:
        torch.nn.init.xavier_uniform(m.weight)
        m.bias.data.fill_(0.01)


data=pd.DataFrame(data)
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.1)
#main_d=X_train.values.reshape(len(X_train),1, 64, order='F')
#main_d_train=np.expand_dims(main_d.astype('float32'), 1)

d1=X_test.values.reshape(len(X_test),1, 1000, order='F')
d1_test=np.expand_dims(d1.astype('float32'), 1)
l1_test=y_test
f1_trn_per=[]
f1_vld_per=[]
f1_test_per=[]
f1_vld_perfold=[]
for per in np.linspace(0.2,1,5):
    print('percentage of train:', per*100)
    indices = np.random.permutation(len(X_train))
    
    ## take 5 %  of training samples
    patch=round(len(indices)*per.astype('float32'))
    batch_dtrain=np.array(X_train)[indices[0:patch]]
    batch_ytrain=np.array(y_train)[indices[0:patch]]
    #X_train, X_test, y_train, y_test = train_test_split(data, label, test_size=0.1)
    #cv = StratifiedKFold(n_splits=10, random_state=42, shuffle=False)

    batch_size=50
    #f1_trn_cv=[]
    #f1_vld_cv=[]
    #f1_vld_perfold=[]
    #f1_test=[]
    #k=1
    #for train_index, test_index in cv.split(batch_dtrain,batch_ytrain):
        #new_net=net
     #   print("fold",k)
    X_train1, X_val1, y_train1, y_val1 = batch_dtrain, batch_dtrain, batch_ytrain, batch_ytrain
    #print(X_train[test_index].shape)
    d=X_train1.reshape(len(X_train1),1, 1000, order='F')
    d1_train=np.expand_dims(d.astype('float32'), 1)
            #l1_train=y_train
    indices = np.random.permutation(len(d1_train))
    d1_train=d1_train[indices]
    l1_train=y_train1[indices]

    d1=X_val1.reshape(len(X_val1),1, 1000, order='F')
    d1_vald=np.expand_dims(d1.astype('float32'), 1)
          # l1_test=y_test
    indices = np.random.permutation(len(d1_vald))
    d1_vald=d1_vald[indices]
    l1_vald=y_val1[indices]
       #liveloss = PlotLosses()


    #    logs = {}
    f1_trn=[]
    f1_vld=[]
         #net1=net
        #optimizer = optim.Adam(new_net.parameters())  

    for epoch in range(2):  # 500 epochs
            #net1=net
        print("epoch",epoch)
        for phase in ['train', 'validation']:
            if phase == 'train':
                running_loss=0 
                for i in range(len(d1_train)//batch_size-1):    ##iteration
                    #  print(i,)
                    s = i*batch_size
                    e = i*batch_size+batch_size

                    inputs = torch.from_numpy(d1_train[s:e])
                    labels = torch.LongTensor(np.array(l1_train[s:e]))

                            # wrap them in Variable
                    inputs, labels = Variable(inputs), Variable(labels)

                            # zero the parameter gradients
                    optimizer.zero_grad()

                            # forward + backward + optimize
                    outputs = net(inputs)

                    loss = criterion(outputs, labels)
                    loss.backward()

                    optimizer.step()
                    running_loss += loss.data.item()
                #    print("Epoch, Loss - {}, {}".format(i, running_loss))
                    del inputs, labels
                        #print('\n')
                       # trn_loss.append(running_loss)
            else: 
                    #r=random.sample(range(1, len(d1_train)), 100)
                  #  print("len(d1_train)",len(d1_train))
                   # print("len(l1_train)",len(l1_train))
                   # trn_acc.append(evaluate(net, d1_train, l1_train, batch_size = 50)) 
                   # tst_acc.append(evaluate(net, d1_vald, l1_vald, batch_size=50)) 
                f1_trn.append(evaluate(net, d1_train, l1_train, batch_size = 128))
                f1_vld.append(evaluate(net, d1_vald, l1_vald, batch_size=128))
                   # out_train=net(torch.from_numpy(d1_train))
                    #out_vald=net(torch.from_numpy(d1_vald))
                    #loss_trn = criterion(out_train, torch.LongTensor(np.array(l1_train)))
                    #loss_vald= criterion(out_vald, torch.LongTensor(np.array(l1_vald)))
                    #trn_loss.append(loss_trn.data.item())
                    #tst_loss.append(loss_vald.data.item())
                print("f1_train",f1_trn)
                print("f1_vald",f1_vld)
                   # print("train_loss",trn_loss)
                   # print("validation_loss",tst_loss)
                   # logs['log_loss_trn'] = loss_trn.append(loss_trn)
                   # logs['log_loss_tst'] = loss_test.append(loss_test)
                  #  logs['tr_accuracy'] = trn_acc[-1]
                  #  logs['tst_accuracy'] = tst_acc[-1]
                    #del net1
        #print("kkkk")
        #del new_net   
       # trn_acc_cv.append(np.mean(trn_acc[0:20]))
       # tst_acc_cv.append(np.mean(tst_acc[0:20]))
       # trn_loss_cv.append(np.mean(trn_loss[0:20]))
       # tst_loss_cv.append(np.mean(tst_loss[0:20]))
    f1_vld_perfold.append(f1_vld)
    f1_trn_per.append(np.mean(f1_trn[9:19]))
    f1_vld_per.append(np.mean(f1_vld[9:19]))
    f1_test_per.append(evaluate(net, d1_test, l1_test, batch_size = 128))
    print("f1 train for per:", per*100, f1_trn_per)
    print("f1 vald for per:", per*100, f1_vld_per)
    print("f1_test for per",per*100,f1_test_per)
    del f1_trn, f1_vld
        #k=k+1
    #net.apply(weights_init)
    net.apply(init_weights)
    #f1_trn_per.append(f1_trn_cv)
    #f1_tst_per.append(f1_tst_cv)
    #tst_acc_per.append(test_acc)
    #print("f1 train for per:", per*100, f1_trn_per)
    #print("f1 tst for per:", per*100, f1_tst_per)
    #print("test acc_for per:", per*100, test_acc)